# 02 — Roaster Hedging Program (Phase 3 deep module)

**Archetype:** a coffee roaster/buyer with a monthly green-coffee purchase schedule (Starbucks FY2025 10-K template: futures/collars on the "C" price as cash-flow hedges).

This notebook orchestrates only — every number comes from `hedging_workbench`:
frozen curve (Phase 1) → hedge program (10-05) → zero-cost collar (10-06) → IFRS 9 effectiveness (10-07) → margin-liquidity stress (10-08) → hedge-ratio comparison vs recent literature (10-09/10-10).

*Honest framing: learning artifact. No compliance claims; regulation mapped, not asserted satisfied. Frozen unofficial Yahoo data.*

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hedging_workbench.carry import load_curve
from hedging_workbench.data.frozen import load, latest_rate
from hedging_workbench.vol import vol_from_frozen
from hedging_workbench.conventions import CONTRACT_LB, lb_to_usd
from hedging_workbench.hedge import build_program, variation_margin
from hedging_workbench.pricing import zero_cost_collar
from hedging_workbench.effectiveness import assess
from hedging_workbench.stress import stress_report
from hedging_workbench.cvar import cvar_optimal_ratio
from hedging_workbench.bekk import fit_bekk

curve = load_curve("coffee")
rets, vol_fit = vol_from_frozen()
f0 = float(curve["price"].iloc[0])
sigma = vol_fit.garch_last / 100.0
r = latest_rate()
print(f"curve date {curve.attrs['curve_date'].date()} | front {curve['label'].iloc[0]} @ {f0:.2f} c/lb")
print(f"GARCH(1,1) working vol {vol_fit.garch_last:.1f}%/yr | SOFR {r:.2%}")

curve date 2026-09-04 | front Sep 2026 @ 324.25 c/lb
GARCH(1,1) working vol 39.2%/yr | SOFR 3.66%


## 1. Exposure, contract selection, roll schedule

12 monthly purchases of 100,000 lb (≈ 2.67 contracts each). Selection rule: a month's purchase is hedged with **that month's delivery contract** when one exists (lock the C-price, offset before last trading day); months without a chain contract map forward to the next expiry. Rolls exit 10 calendar days before expiry.

In [2]:
prog = build_program(100_000, "2026-09-01", 12, universe="coffee")
prog.exposure[["month", "volume_lb", "contracts", "label", "expiry", "gap"]]

,month,volume_lb,contracts,label,expiry,gap
0,2026-09-01,100000.0,2.666667,Sep 2026,2026-09-15,False
1,2026-10-01,100000.0,2.666667,Dec 2026,2026-12-15,False
2,2026-11-01,100000.0,2.666667,Dec 2026,2026-12-15,False
3,2026-12-01,100000.0,2.666667,Dec 2026,2026-12-15,False
4,2027-01-01,100000.0,2.666667,Mar 2027,2027-03-15,False
5,2027-02-01,100000.0,2.666667,Mar 2027,2027-03-15,False
6,2027-03-01,100000.0,2.666667,Mar 2027,2027-03-15,False
7,2027-04-01,100000.0,2.666667,May 2027,2027-05-15,False
8,2027-05-01,100000.0,2.666667,May 2027,2027-05-15,False
9,2027-06-01,100000.0,2.666667,Jul 2027,2027-07-15,False


In [3]:
prog.rolls

,roll_date,front_symbol,front_label,front_expiry,back_symbol,back_label
0,2026-09-05,KCU26.NYB,Sep 2026,2026-09-15,KCZ26.NYB,Dec 2026
1,2026-12-05,KCZ26.NYB,Dec 2026,2026-12-15,KCH27.NYB,Mar 2027
2,2027-03-05,KCH27.NYB,Mar 2027,2027-03-15,KCK27.NYB,May 2027
3,2027-05-05,KCK27.NYB,May 2027,2027-05-15,KCN27.NYB,Jul 2027
4,2027-07-05,KCN27.NYB,Jul 2027,2027-07-15,KCU27.NYB,Sep 2027


Every roll date sits before its front expiry, and no month falls past the chain (`gap` all False) — the 8-contract chain covers the full year.

## 2. Zero-cost collar

Long put at the protection level (280 c/lb); the short call strike is solved so the premiums cancel exactly. Vol is the Phase 2 GARCH working vol; T = 0.25y.

In [4]:
collar = zero_cost_collar(f0, put_strike=280.0, t=0.25, sigma=sigma, r=r)
print(f"f0 {f0:.2f} | put {collar.put_strike:.2f} @ {collar.put_premium:.3f} c/lb "
      f"| call {collar.call_strike:.2f} @ {collar.call_premium:.3f} c/lb")
print(f"premium gap {collar.premium_gap:.2e} c/lb  (zero-cost by construction)")

grid = np.linspace(220, 380, 161)
usd_per_cent = lb_to_usd(100_000)   # one month's volume: $ per cent/lb
fut_pnl = (grid - f0) * usd_per_cent
col_pnl = np.array([collar.payoff(g) for g in grid]) * usd_per_cent

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(grid, fut_pnl, label="long futures", lw=2)
ax.plot(grid, col_pnl, label="collar (long put / short call)", lw=2)
ax.axhline(0, color="k", lw=0.5)
for x, ls in [(collar.put_strike, "--"), (f0, ":"), (collar.call_strike, "--")]:
    ax.axvline(x, color="gray", ls=ls, lw=0.8)
ax.set_xlabel("coffee C price at expiry (c/lb)"); ax.set_ylabel("P&L (USD, one month)")
ax.set_title("Long futures vs zero-cost collar — one month's purchases")
ax.legend(); fig.tight_layout(); fig

f0 324.25 | put 280.00 @ 7.692 c/lb | call 382.06 @ 7.692 c/lb
premium gap 1.33e-14 c/lb  (zero-cost by construction)


The floor caps the worst month at the put protection; the cap gives up upside beyond the call strike — **zero net premium**. The trade a roaster actually wants: bounded cost, no cash outlay for the option.

## 3. IFRS 9 effectiveness (principles-based, not the 80–125% band)

Assessment window: the program's first three months on a synthetic-but-shaped path. IFRS 9 asks three questions — does the economic relationship hold, does credit dominate the instrument, is the hedge ratio consistent — and the 80–125% dollar-offset band is explicitly **not** the test.

In [5]:
path = np.array([f0, f0 * 1.03, f0 * 0.97, f0 * 1.01])
item = -(np.diff(path, prepend=path[0]) * usd_per_cent)   # buyer: price rise = loss
fut  =  (np.diff(path, prepend=path[0]) * usd_per_cent)
res = assess(hypothetical_pnl=fut, hedged_item_pnl=item,
             designated_ratio=1.0, actual_ratio=prog.exposure["contracts"].mean()
             / (prog.exposure["volume_lb"].mean() / CONTRACT_LB))
pd.Series({"correlation (|r|)": res.correlation,
           "dollar offset": res.dollar_offset,
           "credit share": res.credit_share,
           "ratio deviation": res.ratio_deviation,
           "effective": res.effective}, name="assessment")

correlation (|r|)     1.0
dollar offset         1.0
credit share          0.0
ratio deviation       0.0
effective            True
Name: assessment, dtype: object

**Why not the band:** IAS 39's mechanical 80–125% dollar-offset could pass a hedge whose cash flows only *coincidentally* netted (timing mismatch) and fail a genuinely effective one whose ratio sat outside the band. IFRS 9.6.3.2 replaced it with principles. The module's boundary tests (in `tests/test_effectiveness.py`) construct both cases: in-band-but-uncorrelated **fails**, outside-band-but-aligned **passes**.

The 1-page designation memo (`memos/ifrs9_designation_memo.md`) mirrors the Starbucks 10-K structure — cash-flow hedge designation, AOCI treatment, margin collateral — and maps EMIR 3 NFC thresholds to this volume (~€3–4M notional, ~0.1% of the €3B threshold: below-threshold NFC, hedging exemption relevant).

## 4. Margin-liquidity stress (FSB 2024 framing)

Scenarios apply a constant annual shock over one quarter to the frozen path; shocks derive from the GARCH vol. Peak funding need = worst drawdown of the margin balance below its starting level.

In [6]:
kc = load(["KC=F"])["KC=F"].dropna()
report = stress_report(prog.exposure["contracts"].iloc[0], kc, sigma,
                       collar=collar, collateral_usd=37.9e6)   # 10-K margin reference
rows = [{"scenario": s.scenario, "shock/yr": s.shock_annual,
         "futures peak $": s.peak_need_usd,
         "collar peak $": s.collar_peak_need_usd,
         "relief $": s.peak_need_usd - (s.collar_peak_need_usd or 0),
         "buffer vs $37.9M": s.buffer_usd} for s in report]
pd.DataFrame(rows)

,scenario,shock/yr,futures peak $,collar peak $,relief $,buffer vs $37.9M
0,down_1sigma,-0.392424,30300.218964,30300.218964,0.000000,3.786970e+07
1,down_2sigma,-0.784847,57768.970636,44250.000000,13518.970636,3.784223e+07
2,up_2sigma,0.784847,0.000000,0.000000,0.000000,3.790000e+07
3,backwardation_widening,-0.588635,71117.197846,44250.000000,26867.197846,3.782888e+07


In [7]:
names = [s.scenario for s in report]
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, [s.peak_need_usd for s in report], 0.4, label="long futures")
ax.bar(x + 0.2, [s.collar_peak_need_usd for s in report], 0.4, label="collar")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("peak margin call (USD, quarter)")
ax.set_title("Margin-liquidity stress: futures vs collar (put monetised daily — stated simplification)")
ax.legend(); fig.tight_layout(); fig

The collar's peak funding need is bounded by the strike protection in every down scenario — the FSB's point: derivatives that hedge price risk still create liquidity risk, and option structures change that profile, not the hedge effectiveness.

## 5. Hedge-ratio comparison: program vs recent literature

Three answers to "how much to hedge", on the same frozen returns (KC=F item vs KCZ26 futures):

| method | objective | source |
|---|---|---|
| program | 1.0 by construction (volume / contract size) | Starbucks 10-K archetype |
| discrete-CVaR | minimise CVaR of hedged losses | MDPI Mathematics (2025) |
| GARCH-BEKK | time-varying mean-variance (Kroner–Sultan ratio) | BEKK hedge-ratio literature |

In [8]:
kcz = load(["KCZ26.NYB"])["KCZ26.NYB"].dropna()
j = kc.to_frame("kc").join(kcz.rename("kcz"), how="inner").pct_change().dropna()

cvar_res = cvar_optimal_ratio(j["kc"], j["kcz"])
bekk_res = fit_bekk(j)

comparison = pd.DataFrame({
    "method": ["program", "discrete-CVaR (MDPI 2025)", "GARCH-BEKK mean (Kroner-Sultan)"],
    "hedge ratio": [1.0, cvar_res.ratio, bekk_res.hedge_ratio().mean()],
    "objective": ["cash-flow smoothing (volume match)",
                  f"min CVaR@{cvar_res.alpha:.0%} of hedged losses",
                  "min conditional variance"],
})
comparison

,method,hedge ratio,objective
0,program,1.000000,cash-flow smoothing (volume match)
1,discrete-CVaR (MDPI 2025),1.060000,min CVaR@95% of hedged losses
2,GARCH-BEKK mean (Kroner-Sultan),2.493502,min conditional variance


**Verdict.** The CVaR ratio is the variance-minimising answer for a self-contained speculator; the program's 1.0 is the operationally correct answer for a roaster with *known physical volume* — under-hedging leaves C-price exposure on the exact tonnes it must buy, over-hedging turns the hedge into a speculative position (and the designation memo would fail IFRS 9's consistency principle). The BEKK average lands near the CVR/CVaR region and its time-variation is the honest caveat: a static 1.0 program is robust to it *because the objective is cash-flow certainty, not variance minimisation*. Evidence-based choice, stated objective, not asserted.

## 6. Close

- Deliverables this phase: tested package modules (`hedge`, `pricing`, `effectiveness`, `stress`, `cvar`, `bekk`), the designation memo, this notebook.
- Known limits: frozen unofficial data; crude backwardation-widening scenario; put assumed daily-monetisable in stress; thin identification in the S-S fit (Phase 2) carries into the vol input.
- Next: Phase 4 — structured participation note on the same underlying.